In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/florida_restaurant_inspections.csv.gz",
    parse_dates=["Inspection Date"],
    low_memory=False
)

In [ ]:
(
    df[df["License ID"] != 0]
    .groupby(
        [
            "License ID",
            "Business (DBA-Does Business As) Name",
            "Location Address",
            "Location City"
        ]
    )
    .size()
    .sort_values(ascending=False)
    .head(30)
)

License ID  Business (DBA-Does Business As) Name  Location Address    Location City
1           JERSEY MIKE'S SUBS                    9700  DEER LAKE CT  JACKSONVILLE     1
dtype: int64

In [17]:
print(df["License ID"].describe())

print()

print(df["License ID"].value_counts().head(20))

count    681641.000000
mean          0.000001
std           0.001211
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: License ID, dtype: float64

License ID
0    681640
1         1
Name: count, dtype: int64


In [14]:
restaurant_summary = (
    df.groupby(
        [
            "License ID",
            "Business (DBA-Does Business As) Name",
            "Location Address",
            "Location City",
            "County Name"
        ],
        dropna=False
    )
    .agg(
        inspection_count=("Inspection Date", "size"),
        high_priority_violations=(
            "Number of High Priority Violations",
            "sum"
        )
    )
    .reset_index()
)

restaurant_summary.sort_values(
    ["inspection_count", "high_priority_violations"],
    ascending=False
).head(30)

,License ID,Business (DBA-Does Business As) Name,Location Address,Location City,County Name,inspection_count,high_priority_violations
65397,0,MOM'S OG,1017 W UNIVERSITY AVE,GAINESVILLE,Alachua,65,95.0
57290,0,LITTLE INDIA,25000 US HWY 19 N,CLEARWATER,Pinellas,51,114.0
19234,0,CHINA MASTER,"1700 W INTERNATIONAL SPEEDWAY BLVD, SUITE 148",DAYTONA BEACH,Volusia,48,98.0
63837,0,MIA MARKET,140 NE 39 ST #241,MIAMI,Dade,46,46.0
83514,0,SANTA FE MEXICAN GRILL,800 CLEARWATER LARGO RD,LARGO,Pinellas,43,122.0
15874,0,CASA DORA ITALIAN CAFE,108 E FORSYTH ST,JACKSONVILLE,Duval,42,122.0
9954,0,BLUE ANCHOR,804 E ATLANTIC AVE,DELRAY BEACH,Palm Beach,41,95.0
66131,0,MR HAN RESTAURANT,6944 NW 10 PL,GAINESVILLE,Alachua,40,74.0
69171,0,OCEAN BREEZE BAR & GRILL,521 FLAGLER AVE,NEW SMYRNA BEACH,Volusia,39,85.0
100157,0,THE PARCHED OAK,145 N WOODLAND BLVD,DELAND,Volusia,39,62.0


In [15]:
restaurant_summary["high_priority_per_inspection"] = (
    restaurant_summary["high_priority_violations"]
    / restaurant_summary["inspection_count"]
)

restaurant_summary[
    restaurant_summary["inspection_count"] >= 5
].sort_values(
    "high_priority_per_inspection",
    ascending=False
).head(30)

,License ID,Business (DBA-Does Business As) Name,Location Address,Location City,County Name,inspection_count,high_priority_violations,high_priority_per_inspection
59559,0,MAIN GATE FLEA MARKET FOOD COURT #1,5407 W IRLO BRONSON MEMORIAL HWY,KISSIMMEE,Osceola,10,72.0,7.200000
60806,0,MARIA GUADALUPE GUTIERREZ,10014 ATLANTIC BLVD,JACKSONVILLE,Duval,14,96.0,6.857143
92194,0,SUSHI YAMI,6177 JOG RD,LAKE WORTH,Palm Beach,5,34.0,6.800000
41539,0,HABIBI LEBANESE GRILL,8001 S ORANGE BLOSSOM RD STE 1510,ORLANDO,Orange,5,33.0,6.600000
67050,0,NAME VIET BISTRO MACHI,8150 N 49 ST,PINELLAS PARK,Pinellas,26,161.0,6.192308
28236,0,DON PEPPER'S MEXICAN GRILL & CANTINA,794 S ATLANTIC AVE,ORMOND BEACH,Volusia,7,42.0,6.000000
77046,0,POPPIES RESTAURANT & DELI,4900 LINTON BLVD,DELRAY BEACH,Palm Beach,8,46.0,5.750000
108507,0,WILD WING CAFE,4555 SOUTHSIDE BLVD,JACKSONVILLE,Duval,5,28.0,5.600000
92185,0,SUSHI YAMA,10260 FOREST HILL BLVD,WELLINGTON,Palm Beach,9,49.0,5.444444
80620,0,RIVALES TAQUERIA AND CRAFT BAR,11924 W FOREST HILL BLVD STE 28,WELLINGTON,Palm Beach,8,43.0,5.375000


In [4]:
(
    df.groupby("License ID")
      ["Number of High Priority Violations"]
      .sum()
      .eq(0)
      .value_counts()
)

Number of High Priority Violations
False    2
Name: count, dtype: int64

In [5]:
df["Inspection Type"].value_counts()

Inspection Type
Routine - Food               567371
Complaint Full                55960
Food-Licensing Inspection     50859
Complaint Partial              7398
Disaster Response                28
Routine - Lodging                25
Name: count, dtype: int64

In [6]:
df["Inspection Disposition"].value_counts()

Inspection Disposition
Inspection Completed - No Further Action    420282
Call Back - Complied                         87544
Warning Issued                               85608
Administrative complaint recommended         37932
Call Back - Admin. complaint recommended     17913
Call Back - Extension given, pending         15907
Emergency order recommended                   6782
Emergency Order Callback Complied             4907
Emergency Order Callback Not Complied         3103
Emergency Order Callback Time Extension       1341
Administrative determination recommended        98
Assigned to Inspector                           90
Admin. Complaint Callback Complied              43
Admin. Complaint Callback Not Complied          26
Administrative Complaint Time Extension         22
Not available electronically                    20
Insp. Completed - Warning Given, Pending        18
Allegation Not Observed                          3
Routine Inspection                               2
Name: co

In [7]:
(
    df["County Name"]
      .value_counts()
      .head(20)
)

County Name
Dade            92258
Broward         60813
Orange          56202
Palm Beach      46906
Hillsborough    45341
Duval           35353
Pinellas        34106
Lee             25246
Volusia         19698
Brevard         18114
Polk            15443
Sarasota        14988
Collier         13912
Seminole        12601
Osceola         11847
Pasco           11624
Manatee         10863
St. Johns       10588
Lake             9646
Marion           9276
Name: count, dtype: int64

In [11]:
restaurant = "DUNKIN DONUTS"   # put one restaurant name here

(
    df[df["Business (DBA-Does Business As) Name"] == restaurant]
    .sort_values("Inspection Date")
)

,District,County Number,County Name,License Type Code,License Number,Business (DBA-Does Business As) Name,Location Address,Location City,Location Zip Code,Inspection Number,...,Violation 52,Violation 53,Violation 54,Violation 55,Violation 56,Violation 57,Violation 58,License ID,Inspection Visit ID,fiscal_year
68558,D2,60,Palm Beach,2010,6021684,DUNKIN DONUTS,5641 OKEECHOBEE BLVD,WEST PALM BEACH,33417,3121181,...,0,0,0,0,0,0,0,0,6588713,2021-22
68817,D4,59,Osceola,2010,5911371,DUNKIN DONUTS,1421 S NARCOOSSEE RD,ST. CLOUD,34771,3086749,...,0,0,0,0,0,0,0,0,7244140,2021-22
68341,D1,23,Dade,2010,2334748,DUNKIN DONUTS,2360 W 68 ST #101,HIALEAH,33016,3092837,...,0,0,0,0,0,0,0,0,7594959,2021-22
68246,D2,16,Broward,2010,1623461,DUNKIN DONUTS,751 E COMMERCIAL BLVD,OAKLAND PARK,33334,3096048,...,0,0,0,0,0,0,0,0,6352834,2021-22
67959,D2,16,Broward,2010,1623461,DUNKIN DONUTS,751 E COMMERCIAL BLVD,OAKLAND PARK,33334,3096048,...,0,0,0,0,0,0,0,0,6352834,2021-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
618357,D2,16,Broward,2010,1624167,DUNKIN DONUTS,5141 S UNIVERSITY DR.,DAVIE,33328,3700788,...,0,0,0,0,0,0,0,0,6498176,2025-26
617844,D2,60,Palm Beach,2010,6023352,DUNKIN DONUTS,11575 US HWY 1 STE 85,NORTH PALM BEACH,33408,3692660,...,0,0,0,0,0,0,0,0,9840644,2025-26
616895,D7,21,Collier,2010,2103334,DUNKIN DONUTS,1558 LAKE TRAFFORD RD #7,IMMOKALEE,34142,3701276,...,0,0,0,0,0,0,0,0,6415714,2025-26
616783,D4,15,Brevard,2010,1506721,DUNKIN DONUTS,4525 W NEW HAVEN AVE,WEST MELBOURNE,32904,3700011,...,0,0,0,0,0,0,0,0,6692835,2025-26
